**Scrapping Cloud Docs**

In [1]:
# Cellule 1 — ✅ Vérification des dépendances
# ==========================================
# On vérifie que le nouveau kernel fonctionne bien
# Aucune installation nécessaire, tout est dans le venv

import sys
print(f"🐍 Python    : {sys.version}")
print(f"📍 Venv      : {sys.prefix}\n")

import numpy as np
print(f"✅ numpy     : {np.__version__}")

import pandas as pd
print(f"✅ pandas    : {pd.__version__}")

import aiohttp
print(f"✅ aiohttp   : {aiohttp.__version__}")

import bs4
print(f"✅ bs4       : {bs4.__version__}")

from datetime import datetime
print(f"\n🕐 Date      : {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC")
print("\n🎉 Toutes les dépendances sont OK !")

🐍 Python    : 3.10.11 | packaged by conda-forge | (main, May 10 2023, 18:58:44) [GCC 11.3.0]
📍 Venv      : /mnt/batch/tasks/shared/LS_root/mounts/clusters/dev-station-skander/code/Users/Skander.Kammoun/Agent_Tuteur_Subul/.venv

✅ numpy     : 2.2.6
✅ pandas    : 2.3.3
✅ aiohttp   : 3.13.3
✅ bs4       : 4.14.3

🕐 Date      : 2026-02-17 12:32:15 UTC

🎉 Toutes les dépendances sont OK !


In [2]:
# Cellule 2 — 📚 Imports
# ==========================================
# Tous les imports nécessaires pour le projet
import os                          # Gestion des fichiers et dossiers
import json                        # Lecture/écriture JSON
import asyncio                     # Programmation asynchrone
import aiohttp                     # Requêtes HTTP async
import hashlib                     # Hash MD5 pour détecter les changements
import logging                     # Logs pour suivre l'exécution
import time                        # Mesure du temps
import pandas as pd                # Tableaux de données
from datetime import datetime, timedelta  # Dates et durées
from bs4 import BeautifulSoup      # Parsing HTML
from urllib.parse import urlparse  # Analyse des URLs
from typing import Optional        # Typage

# Configuration du logging : affiche les logs dans le notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

print("✅ Tous les imports sont chargés")

✅ Tous les imports sont chargés


In [3]:
# Cellule 3 — 📁 Configuration des chemins
# ==========================================
# On détecte automatiquement le répertoire de travail
# puis on vérifie que tous les dossiers existent

# Détection automatique du répertoire courant
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")

print(f"📍 Répertoire de travail : {BASE_DIR}")
print(f"📍 Répertoire data       : {DATA_DIR}")

# Les 3 providers cloud
PROVIDERS = ["aws", "azure", "gcp"]

# Créer les dossiers manquants (au cas où)
for provider in PROVIDERS:
    os.makedirs(os.path.join(DATA_DIR, "raw", provider), exist_ok=True)
    os.makedirs(os.path.join(DATA_DIR, "processed", provider), exist_ok=True)
    os.makedirs(os.path.join(DATA_DIR, "vector_store", provider), exist_ok=True)

os.makedirs(os.path.join(DATA_DIR, "logs"), exist_ok=True)

# Afficher la structure
print("\n📂 Structure des dossiers :")
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, "").count(os.sep)
    indent = "   " * level
    folder_name = os.path.basename(root)
    file_count = len(files)
    print(f"{indent}📁 {folder_name}/ ({file_count} fichiers)")

print("\n✅ Structure vérifiée")

📍 Répertoire de travail : /mnt/batch/tasks/shared/LS_root/mounts/clusters/dev-station-skander/code/Users/Skander.Kammoun/Agent_Tuteur_Subul/01_Knowledge_Base
📍 Répertoire data       : /mnt/batch/tasks/shared/LS_root/mounts/clusters/dev-station-skander/code/Users/Skander.Kammoun/Agent_Tuteur_Subul/01_Knowledge_Base/data

📂 Structure des dossiers :
📁 data/ (0 fichiers)
   📁 logs/ (0 fichiers)
   📁 processed/ (0 fichiers)
      📁 aws/ (0 fichiers)
      📁 azure/ (0 fichiers)
      📁 gcp/ (0 fichiers)
   📁 raw/ (0 fichiers)
      📁 aws/ (0 fichiers)
      📁 azure/ (0 fichiers)
      📁 gcp/ (0 fichiers)
   📁 vector_store/ (0 fichiers)
      📁 aws/ (0 fichiers)
      📁 azure/ (0 fichiers)
      📁 gcp/ (0 fichiers)

✅ Structure vérifiée


In [4]:
# Cellule 4 — 🌐 Configuration des sources Cloud
# ==========================================
# Ici on définit TOUTES les pages à scraper pour chaque provider
# Pour chaque provider on a :
#   - base_url         : URL de base de la documentation
#   - target_pages     : Liste des pages spécifiques à scraper
#   - content_selector : Sélecteur CSS pour extraire le contenu principal
#   - title_selector   : Sélecteur CSS pour extraire le titre

CLOUD_SOURCES = {
    # ─────────────────────────────────────────────
    # AWS - Amazon Web Services
    # Source : docs.aws.amazon.com
    # ─────────────────────────────────────────────
    "aws": {
        "base_url": "https://docs.aws.amazon.com/",
        "target_pages": [
            # 🖥️ Compute
            "https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/concepts.html",
            "https://docs.aws.amazon.com/lambda/latest/dg/welcome.html",
            "https://docs.aws.amazon.com/elasticbeanstalk/latest/dg/Welcome.html",
            # 💾 Storage
            "https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html",
            "https://docs.aws.amazon.com/efs/latest/ug/whatisefs.html",
            # 🗄️ Database
            "https://docs.aws.amazon.com/AmazonRDS/latest/UserGuide/Welcome.html",
            "https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/Introduction.html",
            # 🌐 Networking
            "https://docs.aws.amazon.com/vpc/latest/userguide/what-is-amazon-vpc.html",
            "https://docs.aws.amazon.com/Route53/latest/DeveloperGuide/Welcome.html",
            # 📦 Containers
            "https://docs.aws.amazon.com/AmazonECS/latest/developerguide/Welcome.html",
            "https://docs.aws.amazon.com/eks/latest/userguide/what-is-eks.html",
            # 🔐 Security
            "https://docs.aws.amazon.com/IAM/latest/UserGuide/introduction.html",
            # 🤖 AI/ML
            "https://docs.aws.amazon.com/sagemaker/latest/dg/whatis.html",
        ],
        "content_selector": "#main-col-body",
        "title_selector": "h1",
    },

    # ─────────────────────────────────────────────
    # AZURE - Microsoft Azure
    # Source : learn.microsoft.com
    # ─────────────────────────────────────────────
    "azure": {
        "base_url": "https://learn.microsoft.com/en-us/azure/",
        "target_pages": [
            # 🖥️ Compute
            "https://learn.microsoft.com/en-us/azure/virtual-machines/overview",
            "https://learn.microsoft.com/en-us/azure/azure-functions/functions-overview",
            "https://learn.microsoft.com/en-us/azure/app-service/overview",
            # 💾 Storage
            "https://learn.microsoft.com/en-us/azure/storage/blobs/storage-blobs-introduction",
            "https://learn.microsoft.com/en-us/azure/storage/files/storage-files-introduction",
            # 🗄️ Database
            "https://learn.microsoft.com/en-us/azure/azure-sql/database/sql-database-paas-overview",
            "https://learn.microsoft.com/en-us/azure/cosmos-db/introduction",
            # 🌐 Networking
            "https://learn.microsoft.com/en-us/azure/virtual-network/virtual-networks-overview",
            "https://learn.microsoft.com/en-us/azure/load-balancer/load-balancer-overview",
            # 📦 Containers
            "https://learn.microsoft.com/en-us/azure/aks/what-is-aks",
            "https://learn.microsoft.com/en-us/azure/container-instances/container-instances-overview",
            # 🔐 Security
            "https://learn.microsoft.com/en-us/entra/fundamentals/whatis",
            # 🤖 AI
            "https://learn.microsoft.com/en-us/azure/ai-services/openai/overview",
            "https://learn.microsoft.com/en-us/azure/machine-learning/overview-what-is-azure-machine-learning",
        ],
        "content_selector": "main#main",
        "title_selector": "h1",
    },

    # ─────────────────────────────────────────────
    # GCP - Google Cloud Platform
    # Source : cloud.google.com
    # ─────────────────────────────────────────────
    "gcp": {
        "base_url": "https://cloud.google.com/docs",
        "target_pages": [
            # 🖥️ Compute
            "https://cloud.google.com/compute/docs/overview",
            "https://cloud.google.com/functions/docs/concepts/overview",
            "https://cloud.google.com/run/docs/overview/what-is-cloud-run",
            # 💾 Storage
            "https://cloud.google.com/storage/docs/introduction",
            "https://cloud.google.com/filestore/docs/overview",
            # 🗄️ Database
            "https://cloud.google.com/sql/docs/introduction",
            "https://cloud.google.com/spanner/docs/overview",
            "https://cloud.google.com/firestore/docs/overview",
            # 🌐 Networking
            "https://cloud.google.com/vpc/docs/overview",
            "https://cloud.google.com/load-balancing/docs/load-balancing-overview",
            # 📦 Containers
            "https://cloud.google.com/kubernetes-engine/docs/concepts/kubernetes-engine-overview",
            # 🔐 Security
            "https://cloud.google.com/iam/docs/overview",
            # 🤖 AI
            "https://cloud.google.com/vertex-ai/docs/start/introduction-unified-platform",
        ],
        "content_selector": "article.devsite-article",
        "title_selector": "h1",
    },
}

# Résumé
total_pages = sum(len(v["target_pages"]) for v in CLOUD_SOURCES.values())
print(f"✅ Configuration chargée : {total_pages} pages à scraper\n")
for provider, config in CLOUD_SOURCES.items():
    print(f"   ☁️  {provider.upper():6s} → {len(config['target_pages']):2d} pages | Source : {config['base_url']}")

✅ Configuration chargée : 40 pages à scraper

   ☁️  AWS    → 13 pages | Source : https://docs.aws.amazon.com/
   ☁️  AZURE  → 14 pages | Source : https://learn.microsoft.com/en-us/azure/
   ☁️  GCP    → 13 pages | Source : https://cloud.google.com/docs


In [5]:
# Cellule 5 — 🔧 Classe CloudDocsScraper
# ==========================================
# C'est le moteur principal du scraping
# Il gère :
#   - Les requêtes HTTP asynchrones (rapide, non-bloquant)
#   - L'extraction du contenu HTML → texte structuré
#   - La détection de changements (hash MD5)
#   - La sauvegarde en JSON
#   - Le rate limiting (pour ne pas se faire bloquer)

class CloudDocsScraper:
    """
    Scraper asynchrone pour les documentations cloud officielles.
    
    Fonctionnalités :
    - Scraping parallèle avec contrôle de concurrence
    - Détection de changements via hash MD5
    - Extraction structurée (titre, sections, texte complet)
    - Rate limiting pour respecter les serveurs
    - Rapports détaillés
    """

    def __init__(self, output_dir: str, max_concurrent: int = 3, delay: float = 2.0):
        """
        Args:
            output_dir    : Dossier de sortie (data/raw)
            max_concurrent: Nombre max de requêtes simultanées
            delay         : Délai en secondes entre chaque requête
        """
        self.output_dir = output_dir
        self.max_concurrent = max_concurrent
        self.delay = delay
        self.semaphore = asyncio.Semaphore(max_concurrent)

        # Compteurs de statistiques
        self.stats = {"success": 0, "failed": 0, "skipped": 0}

        # Stockage des résultats pour affichage dans le notebook
        self.results_data = []

    # ─────────────────────────────────────────────
    # Méthodes utilitaires
    # ─────────────────────────────────────────────

    @staticmethod
    def _slugify(url: str) -> str:
        """
        Convertit une URL en nom de fichier sûr.
        Ex: 'https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/concepts.html'
            → 'AWSEC2_latest_UserGuide_concepts.html'
        """
        path = urlparse(url).path.strip("/").replace("/", "_")
        return path if path else "index"

    @staticmethod
    def _hash_content(content: str) -> str:
        """
        Génère un hash MD5 du contenu.
        Sert à détecter si le contenu a changé depuis le dernier scraping.
        """
        return hashlib.md5(content.encode("utf-8")).hexdigest()

    def _get_output_path(self, provider: str, url: str) -> str:
        """Retourne le chemin complet du fichier JSON de sortie."""
        provider_dir = os.path.join(self.output_dir, provider)
        os.makedirs(provider_dir, exist_ok=True)
        filename = self._slugify(url) + ".json"
        return os.path.join(provider_dir, filename)

    # ─────────────────────────────────────────────
    # Extraction du contenu HTML
    # ─────────────────────────────────────────────

    def _extract_content(
        self,
        html: str,
        content_selector: str,
        title_selector: str,
        url: str,
    ) -> Optional[dict]:
        """
        Parse le HTML brut et extrait le contenu structuré.
        
        Retourne un dict avec :
        - title        : Titre de la page
        - url          : URL source
        - sections     : Liste de {heading, content}
        - full_text    : Texte brut complet
        - content_hash : Hash MD5 pour détecter les changements
        - char_count   : Nombre de caractères
        - scraped_at   : Timestamp du scraping
        """
        soup = BeautifulSoup(html, "lxml")

        # 1. Extraire le titre
        title_tag = soup.select_one(title_selector)
        title = title_tag.get_text(strip=True) if title_tag else "Sans titre"

        # 2. Extraire le contenu principal via le sélecteur CSS
        content_tag = soup.select_one(content_selector)
        if not content_tag:
            # Fallback : essayer des sélecteurs génériques
            content_tag = soup.select_one("main") or soup.select_one("article")
        if not content_tag:
            logger.error(f"❌ Aucun contenu trouvé pour {url}")
            return None

        # 3. Nettoyer le HTML (supprimer les éléments inutiles)
        for tag in content_tag.find_all(
            ["script", "style", "nav", "footer", "header", "aside"]
        ):
            tag.decompose()

        # 4. Découper en sections (h2/h3 = nouvelle section)
        sections = []
        current_section = {"heading": "Introduction", "content": ""}

        for element in content_tag.find_all(
            ["h2", "h3", "p", "ul", "ol", "pre", "code", "table"]
        ):
            if element.name in ["h2", "h3"]:
                if current_section["content"].strip():
                    sections.append(current_section.copy())
                current_section = {
                    "heading": element.get_text(strip=True),
                    "content": "",
                }
            else:
                text = element.get_text(separator="\n", strip=True)
                if text:
                    current_section["content"] += text + "\n\n"

        if current_section["content"].strip():
            sections.append(current_section)

        # 5. Texte brut complet
        full_text = content_tag.get_text(separator="\n", strip=True)

        # 6. Vérifier que le contenu est suffisant
        if not full_text or len(full_text) < 100:
            logger.warning(f"⚠️ Contenu trop court ({len(full_text)} chars) : {url}")
            return None

        return {
            "title": title,
            "url": url,
            "sections": sections,
            "full_text": full_text,
            "content_hash": self._hash_content(full_text),
            "char_count": len(full_text),
            "scraped_at": datetime.utcnow().isoformat(),
        }

    # ─────────────────────────────────────────────
    # Détection de changements
    # ─────────────────────────────────────────────

    def _has_changed(self, output_path: str, new_hash: str) -> bool:
        """
        Compare le hash du nouveau contenu avec celui déjà sauvegardé.
        Retourne True si le contenu est nouveau ou a changé.
        """
        if not os.path.exists(output_path):
            return True
        try:
            with open(output_path, "r", encoding="utf-8") as f:
                existing = json.load(f)
            return existing.get("content_hash") != new_hash
        except (json.JSONDecodeError, KeyError):
            return True

    # ─────────────────────────────────────────────
    # Scraping d'une page unique
    # ─────────────────────────────────────────────

    async def _scrape_page(
        self,
        session: aiohttp.ClientSession,
        url: str,
        provider: str,
        config: dict,
    ) -> Optional[str]:
        """Scrape une seule page avec rate limiting et gestion d'erreurs."""
        async with self.semaphore:
            try:
                headers = {
                    "User-Agent": (
                        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                        "AppleWebKit/537.36 (KHTML, like Gecko) "
                        "Chrome/120.0.0.0 Safari/537.36"
                    ),
                    "Accept": "text/html,application/xhtml+xml",
                    "Accept-Language": "en-US,en;q=0.9",
                }

                logger.info(f"[{provider.upper()}] 🔄 Scraping : {url}")

                async with session.get(
                    url,
                    headers=headers,
                    timeout=aiohttp.ClientTimeout(total=30),
                ) as response:
                    if response.status != 200:
                        logger.error(f"[{provider.upper()}] ❌ HTTP {response.status} → {url}")
                        self.stats["failed"] += 1
                        return None
                    html = await response.text()

                data = self._extract_content(
                    html=html,
                    content_selector=config["content_selector"],
                    title_selector=config["title_selector"],
                    url=url,
                )

                if not data:
                    self.stats["failed"] += 1
                    return None

                data["provider"] = provider
                output_path = self._get_output_path(provider, url)

                if not self._has_changed(output_path, data["content_hash"]):
                    logger.info(f"[{provider.upper()}] ⏭️ Inchangé : {data['title']}")
                    self.stats["skipped"] += 1
                    return None

                with open(output_path, "w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)

                logger.info(
                    f"[{provider.upper()}] ✅ Sauvegardé : {data['title']} "
                    f"({data['char_count']:,} chars, {len(data['sections'])} sections)"
                )

                self.stats["success"] += 1
                self.results_data.append({
                    "provider": provider.upper(),
                    "title": data["title"],
                    "chars": data["char_count"],
                    "sections": len(data["sections"]),
                    "url": url,
                })

                await asyncio.sleep(self.delay)
                return output_path

            except asyncio.TimeoutError:
                logger.error(f"[{provider.upper()}] ⏰ Timeout : {url}")
                self.stats["failed"] += 1
                return None
            except Exception as e:
                logger.error(f"[{provider.upper()}] ❌ Erreur : {url} → {e}")
                self.stats["failed"] += 1
                return None

    # ─────────────────────────────────────────────
    # Scraping d'un provider complet
    # ─────────────────────────────────────────────

    async def scrape_provider(self, provider: str) -> list:
        """Scrape toutes les pages d'un provider."""
        if provider not in CLOUD_SOURCES:
            logger.error(f"Provider inconnu : {provider}")
            return []

        config = CLOUD_SOURCES[provider]
        connector = aiohttp.TCPConnector(limit=self.max_concurrent, ssl=False)

        async with aiohttp.ClientSession(connector=connector) as session:
            tasks = [
                self._scrape_page(session, url, provider, config)
                for url in config["target_pages"]
            ]
            results = await asyncio.gather(*tasks)

        saved = [r for r in results if r is not None]
        logger.info(f"[{provider.upper()}] 📊 {len(saved)} pages sauvegardées")
        return saved

    # ─────────────────────────────────────────────
    # Scraping complet (les 3 clouds)
    # ─────────────────────────────────────────────

    async def scrape_all(self) -> dict:
        """Scrape les 3 providers cloud l'un après l'autre."""
        logger.info("=" * 60)
        logger.info("🚀 Démarrage du scraping des documentations cloud")
        logger.info("=" * 60)

        all_results = {}
        for provider in CLOUD_SOURCES:
            logger.info(f"\n{'─' * 50}")
            logger.info(f"☁️  Scraping {provider.upper()}...")
            logger.info(f"{'─' * 50}")
            all_results[provider] = await self.scrape_provider(provider)
            await asyncio.sleep(3)

        logger.info("\n" + "=" * 60)
        logger.info("📊 SCRAPING TERMINÉ")
        logger.info("=" * 60)

        return all_results


print("✅ Classe CloudDocsScraper chargée")

✅ Classe CloudDocsScraper chargée


In [6]:
# Cellule 6 — 🚀 LANCER LE SCRAPING + 🔄 CRON
# ==========================================
# Cette cellule lance le scraping
# 2 modes disponibles :
#   - Mode UNIQUE  : scrape une seule fois (pour tester)
#   - Mode CRON    : scrape à intervalle régulier (production)

# ─────────────────────────────────────────────
# ⚙️ CONFIGURATION — MODIFIER ICI
# ─────────────────────────────────────────────
CRON_CONFIG = {
    "enabled": False,             # False = scraping unique | True = cron activé
    "interval_hours": 24,         # Intervalle entre chaque scraping
    "max_runs": 0,                # 0 = infini | sinon s'arrête après N runs
    "stop_if_no_changes": False,  # True = s'arrête si rien n'a changé
}

# Dossier de sortie
OUTPUT_DIR = os.path.join(DATA_DIR, "raw")


async def run_scraping():
    """Exécute un cycle complet de scraping."""

    scraper = CloudDocsScraper(
        output_dir=OUTPUT_DIR,
        max_concurrent=3,
        delay=2.0
    )

    start_time = time.time()
    results = await scraper.scrape_all()
    duration = time.time() - start_time

    print("\n" + "=" * 60)
    print("📊 RÉSULTATS DU SCRAPING")
    print("=" * 60)
    print(f"   ✅ Succès  : {scraper.stats['success']}")
    print(f"   ⏭️  Ignorés : {scraper.stats['skipped']}")
    print(f"   ❌ Échecs  : {scraper.stats['failed']}")
    print(f"   ⏱️  Durée   : {duration:.1f} secondes")

    # Sauvegarder le rapport
    report = {
        "timestamp": datetime.utcnow().isoformat(),
        "duration_seconds": round(duration, 1),
        "stats": scraper.stats,
        "documents": scraper.results_data,
    }
    report_name = f"scraping_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.json"
    report_path = os.path.join(DATA_DIR, "logs", report_name)
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    print(f"   📄 Rapport : {report_path}")

    return scraper


async def cron_loop():
    """Boucle Cron qui relance le scraping à intervalle régulier."""
    run_count = 0
    interval_seconds = CRON_CONFIG["interval_hours"] * 3600

    while True:
        run_count += 1
        print("\n" + "🔄" * 30)
        print(f"🕐 CRON RUN #{run_count} — {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC")
        print("🔄" * 30)

        scraper = await run_scraping()
        has_changes = scraper.stats["success"] > 0

        if has_changes:
            print(f"\n📢 {scraper.stats['success']} documents mis à jour")
            print("   → Re-indexing RAG nécessaire (Étape 2)")
        else:
            print("\n✅ Aucun changement détecté. RAG à jour.")
            if CRON_CONFIG["stop_if_no_changes"]:
                print("⏹️ Arrêt du cron (aucun changement)")
                break

        if CRON_CONFIG["max_runs"] > 0 and run_count >= CRON_CONFIG["max_runs"]:
            print(f"\n⏹️ Max runs atteint ({CRON_CONFIG['max_runs']})")
            break

        next_run = datetime.utcnow() + timedelta(hours=CRON_CONFIG["interval_hours"])
        print(f"\n⏳ Prochain scraping : {next_run.strftime('%Y-%m-%d %H:%M:%S')} UTC")
        print(f"   (dans {CRON_CONFIG['interval_hours']}h)")
        print("   💡 Pour arrêter : Interrompre le kernel (bouton ⏹️)")

        await asyncio.sleep(interval_seconds)


# ─────────────────────────────────────────────
# 🚀 LANCEMENT
# ─────────────────────────────────────────────
if CRON_CONFIG["enabled"]:
    print("🔄 MODE CRON ACTIVÉ")
    print(f"   📅 Intervalle     : toutes les {CRON_CONFIG['interval_hours']}h")
    print(f"   🔁 Max runs       : {'∞' if CRON_CONFIG['max_runs'] == 0 else CRON_CONFIG['max_runs']}")
    print(f"   ⏹️  Stop si RAS   : {CRON_CONFIG['stop_if_no_changes']}")
    print(f"   💡 Pour arrêter   : Interrompre le kernel\n")
    await cron_loop()
else:
    print("▶️ MODE UNIQUE (scraping une seule fois)\n")
    scraper = await run_scraping()

2026-02-17 12:34:43,491 [INFO] [AWS] ✅ Sauvegardé : What is AWS Elastic Beanstalk? (2,753 chars, 5 sections)
2026-02-17 12:34:43,544 [INFO] [AWS] ✅ Sauvegardé : What is Amazon EC2? (9,598 chars, 7 sections)
2026-02-17 12:34:43,584 [INFO] [AWS] ✅ Sauvegardé : What is AWS Lambda? (4,336 chars, 4 sections)
2026-02-17 12:34:45,496 [INFO] [AWS] 🔄 Scraping : https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html
2026-02-17 12:34:45,546 [INFO] [AWS] 🔄 Scraping : https://docs.aws.amazon.com/efs/latest/ug/whatisefs.html
2026-02-17 12:34:45,585 [INFO] [AWS] 🔄 Scraping : https://docs.aws.amazon.com/AmazonRDS/latest/UserGuide/Welcome.html
2026-02-17 12:34:45,651 [INFO] [AWS] ✅ Sauvegardé : What is Amazon Relational Database Service (Amazon RDS)? (20,449 chars, 24 sections)
2026-02-17 12:34:45,713 [INFO] [AWS] ✅ Sauvegardé : What is Amazon S3? (35,105 chars, 28 sections)
2026-02-17 12:34:45,753 [INFO] [AWS] ✅ Sauvegardé : What is Amazon Elastic File System? (4,654 chars, 2 sections)
202


📊 RÉSULTATS DU SCRAPING
   ✅ Succès  : 38
   ⏭️  Ignorés : 0
   ❌ Échecs  : 2
   ⏱️  Durée   : 43.1 secondes
   📄 Rapport : /mnt/batch/tasks/shared/LS_root/mounts/clusters/dev-station-skander/code/Users/Skander.Kammoun/Agent_Tuteur_Subul/01_Knowledge_Base/data/logs/scraping_20260217_123526.json


In [7]:
# Cellule 7 — 📊 Visualiser les résultats
# ==========================================
# Affiche un tableau avec pandas

if scraper.results_data:
    df = pd.DataFrame(scraper.results_data)

    # Tableau détaillé
    print("📋 Documents scrapés :\n")
    display(df[["provider", "title", "chars", "sections"]])

    # Statistiques par provider
    print("\n📊 Statistiques par provider :\n")
    summary = df.groupby("provider").agg(
        documents=("title", "count"),
        total_chars=("chars", "sum"),
        avg_chars=("chars", "mean"),
        total_sections=("sections", "sum"),
    ).round(0)
    display(summary)

    # Totaux
    print(f"\n📈 TOTAUX :")
    print(f"   📄 Documents  : {len(df)}")
    print(f"   📝 Caractères : {df['chars'].sum():,}")
    print(f"   📑 Sections   : {df['sections'].sum()}")
else:
    print("⚠️ Aucun document scrapé")
    print("   → Soit tout est déjà à jour")
    print("   → Soit il y a des erreurs (vérifiez les logs)")

📋 Documents scrapés :



,provider,title,chars,sections
0,AWS,What is AWS Elastic Beanstalk?,2753,5
1,AWS,What is Amazon EC2?,9598,7
2,AWS,What is AWS Lambda?,4336,4
3,AWS,What is Amazon Relational Database Service (Am...,20449,24
4,AWS,What is Amazon S3?,35105,28
5,AWS,What is Amazon Elastic File System?,4654,2
6,AWS,What is Amazon VPC?,7184,5
7,AWS,What is Amazon DynamoDB?,20051,23
8,AWS,What is Amazon Route 53?,3695,1
9,AWS,What is Amazon Elastic Container Service?,5316,6



📊 Statistiques par provider :



,documents,total_chars,avg_chars,total_sections
provider,,,,
AWS,13,128806,9908.0,117
AZURE,14,209690,14978.0,255
GCP,11,94550,8595.0,115



📈 TOTAUX :
   📄 Documents  : 38
   📝 Caractères : 433,046
   📑 Sections   : 487


In [8]:
# Cellule 8 — 🔍 Inspecter un document
# ==========================================
# Permet de lire le contenu d'un document scrapé
# ⬇️ MODIFIER ICI pour explorer différents fichiers

PROVIDER = "azure"   # Options : "aws", "azure", "gcp"
FILE_INDEX = 0        # 0 = premier fichier, 1 = deuxième, etc.

# Lister les fichiers
raw_dir = os.path.join(DATA_DIR, "raw", PROVIDER)

if os.path.exists(raw_dir):
    files = [f for f in os.listdir(raw_dir) if f.endswith(".json")]

    print(f"📁 {PROVIDER.upper()} — {len(files)} fichiers :\n")
    for i, f in enumerate(files):
        marker = "👉" if i == FILE_INDEX else "  "
        print(f"  {marker} [{i}] {f}")

    if files and FILE_INDEX < len(files):
        filepath = os.path.join(raw_dir, files[FILE_INDEX])
        with open(filepath, "r", encoding="utf-8") as f:
            doc = json.load(f)

        print(f"\n{'=' * 60}")
        print(f"📄 Titre     : {doc['title']}")
        print(f"🔗 URL       : {doc['url']}")
        print(f"☁️  Provider  : {doc['provider'].upper()}")
        print(f"📏 Taille    : {doc['char_count']:,} caractères")
        print(f"📑 Sections  : {len(doc['sections'])}")
        print(f"🕐 Scrapé le : {doc['scraped_at']}")
        print(f"🔑 Hash      : {doc['content_hash']}")

        print(f"\n{'─' * 60}")
        print("📑 SECTIONS :")
        for i, section in enumerate(doc["sections"]):
            print(f"  [{i}] {section['heading']} ({len(section['content']):,} chars)")

        print(f"\n{'─' * 60}")
        print("📝 APERÇU (1000 premiers caractères) :")
        print(f"{'─' * 60}")
        print(doc["full_text"][:1000])
        print("...")
    else:
        print(f"\n⚠️ Pas de fichier à l'index {FILE_INDEX}")
else:
    print(f"⚠️ Dossier {raw_dir} n'existe pas")

📁 AZURE — 14 fichiers :

  👉 [0] en-us_azure_ai-services_openai_overview.json
     [1] en-us_azure_aks_what-is-aks.json
     [2] en-us_azure_app-service_overview.json
     [3] en-us_azure_azure-functions_functions-overview.json
     [4] en-us_azure_azure-sql_database_sql-database-paas-overview.json
     [5] en-us_azure_container-instances_container-instances-overview.json
     [6] en-us_azure_cosmos-db_introduction.json
     [7] en-us_azure_load-balancer_load-balancer-overview.json
     [8] en-us_azure_machine-learning_overview-what-is-azure-machine-learning.json
     [9] en-us_azure_storage_blobs_storage-blobs-introduction.json
     [10] en-us_azure_storage_files_storage-files-introduction.json
     [11] en-us_azure_virtual-machines_overview.json
     [12] en-us_azure_virtual-network_virtual-networks-overview.json
     [13] en-us_entra_fundamentals_whatis.json

📄 Titre     : Foundry Models sold directly by Azure
🔗 URL       : https://learn.microsoft.com/en-us/azure/ai-services/openai/

In [9]:
# Cellule 9 — 📁 Résumé final
# ==========================================
# Vue d'ensemble de tout ce qui a été scrapé

print("📁 RÉSUMÉ DE LA BASE DE DONNÉES\n")
print(f"  {'Provider':<10} {'Fichiers':>10} {'Taille':>15}")
print("  " + "─" * 40)

grand_total_files = 0
grand_total_size = 0

for provider in PROVIDERS:
    raw_dir = os.path.join(DATA_DIR, "raw", provider)
    if os.path.exists(raw_dir):
        files = [f for f in os.listdir(raw_dir) if f.endswith(".json")]
        total_size = sum(
            os.path.getsize(os.path.join(raw_dir, f)) for f in files
        )
        grand_total_files += len(files)
        grand_total_size += total_size
        size_kb = total_size / 1024
        print(f"  {provider.upper():<10} {len(files):>10} {size_kb:>12.1f} KB")
    else:
        print(f"  {provider.upper():<10} {'0':>10} {'0.0':>12} KB")

print("  " + "─" * 40)
print(f"  {'TOTAL':<10} {grand_total_files:>10} {grand_total_size/1024:>12.1f} KB")

print(f"\n✅ Base de données prête pour l'Étape 2 (Chunking + Embeddings)")

📁 RÉSUMÉ DE LA BASE DE DONNÉES

  Provider     Fichiers          Taille
  ────────────────────────────────────────
  AWS                13        305.0 KB
  AZURE              14        481.0 KB
  GCP                11        219.9 KB
  ────────────────────────────────────────
  TOTAL              38       1006.0 KB

✅ Base de données prête pour l'Étape 2 (Chunking + Embeddings)


In [10]:
# Cellule — 📍 Localiser les fichiers scrapés
# ==========================================
import os

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")

print(f"📍 Répertoire de travail : {BASE_DIR}")
print(f"📍 Répertoire data       : {DATA_DIR}")
print(f"📍 Répertoire raw        : {RAW_DIR}")

print("\n" + "=" * 70)
print("📁 FICHIERS SCRAPÉS")
print("=" * 70)

for provider in ["aws", "azure", "gcp"]:
    provider_dir = os.path.join(RAW_DIR, provider)
    
    if os.path.exists(provider_dir):
        files = [f for f in os.listdir(provider_dir) if f.endswith(".json")]
        
        print(f"\n☁️  {provider.upper()} — {len(files)} fichiers")
        print(f"   📂 Chemin : {provider_dir}")
        print(f"   {'─' * 60}")
        
        for i, f in enumerate(files):
            filepath = os.path.join(provider_dir, f)
            size_kb = os.path.getsize(filepath) / 1024
            print(f"   [{i}] {f} ({size_kb:.1f} KB)")
    else:
        print(f"\n☁️  {provider.upper()} — ⚠️ Dossier vide ou inexistant")

# Logs
logs_dir = os.path.join(DATA_DIR, "logs")
if os.path.exists(logs_dir):
    log_files = os.listdir(logs_dir)
    print(f"\n📄 LOGS — {len(log_files)} fichiers")
    print(f"   📂 Chemin : {logs_dir}")
    for f in log_files:
        print(f"   → {f}")

📍 Répertoire de travail : /mnt/batch/tasks/shared/LS_root/mounts/clusters/dev-station-skander/code/Users/Skander.Kammoun/Agent_Tuteur_Subul/01_Knowledge_Base
📍 Répertoire data       : /mnt/batch/tasks/shared/LS_root/mounts/clusters/dev-station-skander/code/Users/Skander.Kammoun/Agent_Tuteur_Subul/01_Knowledge_Base/data
📍 Répertoire raw        : /mnt/batch/tasks/shared/LS_root/mounts/clusters/dev-station-skander/code/Users/Skander.Kammoun/Agent_Tuteur_Subul/01_Knowledge_Base/data/raw

📁 FICHIERS SCRAPÉS

☁️  AWS — 13 fichiers
   📂 Chemin : /mnt/batch/tasks/shared/LS_root/mounts/clusters/dev-station-skander/code/Users/Skander.Kammoun/Agent_Tuteur_Subul/01_Knowledge_Base/data/raw/aws
   ────────────────────────────────────────────────────────────
   [0] amazondynamodb_latest_developerguide_Introduction.html.json (45.6 KB)
   [1] AmazonECS_latest_developerguide_Welcome.html.json (13.7 KB)
   [2] AmazonRDS_latest_UserGuide_Welcome.html.json (48.0 KB)
   [3] AmazonS3_latest_userguide_Welcome